In [19]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeopyError
import geopandas as gpd
import os
from shapely.geometry import Point
pd.set_option('display.max_columns', 200)

In [27]:
healthcare_df = pd.read_excel('../data/interim/healthcare_cleaned.xlsx')
healthcare_df.head()

,_Location_latitude,_Location_longitude,_Location_altitude,_Location_precision,Age,Gender,Marital Status,Children,Employment Status,Monthly Household Income,Health_Insurance,Insurance_cover,Last_Hosptal_visit,Last_Hosptal_visit_insurance,Check-up,Years_to_Checkup,Cancer_Screening,Years_to_Screening
0,-0.715812,37.147506,1361.900024,20.000,18-30,Male,Single,0,Unemployed,Less than 10000,No,NaN,8.0,No,Yes,1,No,NaN
1,-0.715816,37.147508,1361.900024,20.000,41-50,Female,Married,5,Self-employed,20001-30000,No,NaN,6.0,Yes,No,NaN,Yes,4+
2,-0.715708,37.147490,1361.900024,20.000,18-30,Male,Single,0,Self-employed,10001-20000,Yes,NHIF,16.0,Yes,No,NaN,No,NaN
3,-0.715734,37.147480,1361.900024,20.000,41-50,Male,Married,7,Self-employed,20001-30000,Yes,NHIF,13.0,Yes,No,NaN,Yes,4+
4,-0.715804,37.147536,1361.900024,26.107,18-30,Female,Single,0,Unemployed,10001-20000,Yes,NHIF,2.0,Yes,No,NaN,No,NaN


### Generate a Location_name from the longitudes, latitudes

In [32]:
# 1. SETUP YOUR PATHS & SAMPLE DATA
# ==========================================
# File path to your downloaded geoBoundaries or administrative map file
# Supports (.shp, .geojson, .topojson, .json)
MAP_FILE_PATH = r'C:\Users\admin\Desktop\DataSets Projects\Health_Care_Analysis\data\raw\geoBoundaries-KEN-ADM1-all\geoBoundaries-KEN-ADM1.shp'

# Mock DataFrame replicating your exact dataset structure
# Coordinates reflect places in Nairobi and Mombasa, Kenya
print("--- Original Healthcare DataFrame ---")
healthcare_df.head()


# ==========================================
# 2. DATA CLEANING & GEOPANDAS CONVERSION
# ==========================================
# Step A: Filter out rows with missing coordinates to prevent Point generation crashes
valid_coords_mask = healthcare_df['_Location_latitude'].notna() & healthcare_df['_Location_longitude'].notna()
df_clean = healthcare_df[valid_coords_mask].copy()

# Step B: Transform the cleaned pandas rows into geographic spatial points
gdf_data = gpd.GeoDataFrame(
    df_clean, 
    geometry=gpd.points_from_xy(df_clean['_Location_longitude'], df_clean['_Location_latitude']),
    crs="EPSG:4326"  # Standard GPS system (WGS 84)
)


# ==========================================
# 3. LOAD THE OFFLINE REGIONAL MAP
# ==========================================
if not os.path.exists(MAP_FILE_PATH):
    raise FileNotFoundError(
        f"Could not find map file at {MAP_FILE_PATH}. "
        "Please check your relative folder path or file extension!"
    )

print("\n--- Loading Offline Boundary Map ---")
map_regions = gpd.read_file(MAP_FILE_PATH)
map_regions = map_regions.to_crs("EPSG:4326")  # Align map coordinate system with GPS


# ==========================================
# 4. SPATIAL INTERSECT JOIN (OFFLINE FEATURE ENGINEERING)
# ==========================================
# Check which polygon boundary boundary envelope contains each coordinate point
joined_df = gpd.sjoin(gdf_data, map_regions, how="left", predicate="intersects")

# List of typical attribute labels used by various global GIS providers
possible_name_columns = [
    'shapeName',   # geoBoundaries standard
    'ADM1_EN',     # HDX / UN standard (English)
    'NAME_1',      # GADM dataset standard
    'COUNTY',      # Kenya specific mapping standard
    'Name',        # Generic GeoJSON label
    'nam_1'        # Shapefile truncated name fallback
]

# Scan the map columns to locate the valid descriptive name parameter
location_col = next((col for col in possible_name_columns if col in joined_df.columns), None)

# ==========================================
# 5. MERGE ENGINEERED FEATURE PERMANENTLY BACK TO MAIN DF
# ==========================================
if location_col:
    print(f"Success: Mapping names extracted using feature token: '{location_col}'")
    # Isolate the names into a clean Series aligned to the original index
    extracted_names = joined_df[location_col]
else:
    print("\nWarning: Structural fallback triggered.")
    fallback_col = [c for c in map_regions.columns if c not in ['geometry', 'index_right']][0]
    extracted_names = joined_df[fallback_col]

# Convert the series into a DataFrame named 'location_name'
extracted_names_df = extracted_names.to_frame(name='location_name')

# CRITICAL: Overwrite healthcare_df by performing a hard structural left merge
healthcare_df = healthcare_df.merge(
    extracted_names_df, 
    left_index=True, 
    right_index=True, 
    how='left'
)

# Clean up rows that had missing coordinates or fell outside the map boundaries
healthcare_df['location_name'] = healthcare_df['location_name'].fillna("Unknown / Missing Coordinates")


# ==========================================
# 6. VERIFY RESULTS
# ==========================================
print("\n--- Final Appended Healthcare Dataset Columns ---")
healthcare_df.columns.tolist()

print("\n--- Head of your updated dataset ---")
healthcare_df[['_Location_latitude', '_Location_longitude', 'location_name']].head(20)


--- Original Healthcare DataFrame ---

--- Loading Offline Boundary Map ---
Success: Mapping names extracted using feature token: 'shapeName'

--- Final Appended Healthcare Dataset Columns ---

--- Head of your updated dataset ---


,_Location_latitude,_Location_longitude,location_name
0,-0.715812,37.147506,Murang'a
1,-0.715816,37.147508,Murang'a
2,-0.715708,37.147490,Murang'a
3,-0.715734,37.147480,Murang'a
4,-0.715804,37.147536,Murang'a
5,-0.715893,37.147353,Murang'a
6,-0.715940,37.146309,Murang'a
7,-0.715912,37.147077,Murang'a
8,-0.715871,37.146672,Murang'a
9,-0.716643,37.145942,Murang'a


In [38]:
#healthcare_df.drop(columns=['location_name_x','location_name_y'], inplace=True)
healthcare_df.head(10)

,_Location_latitude,_Location_longitude,_Location_altitude,_Location_precision,Age,Gender,Marital Status,Children,Employment Status,Monthly Household Income,Health_Insurance,Insurance_cover,Last_Hosptal_visit,Last_Hosptal_visit_insurance,Check-up,Years_to_Checkup,Cancer_Screening,Years_to_Screening,location_name
0,-0.715812,37.147506,1361.900024,20.000,18-30,Male,Single,0,Unemployed,Less than 10000,No,NaN,8.0,No,Yes,1,No,NaN,Murang'a
1,-0.715816,37.147508,1361.900024,20.000,41-50,Female,Married,5,Self-employed,20001-30000,No,NaN,6.0,Yes,No,NaN,Yes,4+,Murang'a
2,-0.715708,37.147490,1361.900024,20.000,18-30,Male,Single,0,Self-employed,10001-20000,Yes,NHIF,16.0,Yes,No,NaN,No,NaN,Murang'a
3,-0.715734,37.147480,1361.900024,20.000,41-50,Male,Married,7,Self-employed,20001-30000,Yes,NHIF,13.0,Yes,No,NaN,Yes,4+,Murang'a
4,-0.715804,37.147536,1361.900024,26.107,18-30,Female,Single,0,Unemployed,10001-20000,Yes,NHIF,2.0,Yes,No,NaN,No,NaN,Murang'a
5,-0.715893,37.147353,1361.599976,32.483,31-40,Female,Married,2,Self-employed,10001-20000,Yes,NHIF,4.0,Yes,No,NaN,No,NaN,Murang'a
6,-0.715940,37.146309,1362.200073,56.100,18-30,Male,Single,0,Self-employed,Less than 10000,No,NaN,24.0,No,No,NaN,No,NaN,Murang'a
7,-0.715912,37.147077,1362.200073,33.011,31-40,Female,Married,3,Employed,20001-30000,Yes,NHIF,5.0,Yes,No,NaN,No,NaN,Murang'a
8,-0.715871,37.146672,1362.200073,52.400,31-40,Female,Married,2,Employed,10001-20000,Yes,NHIF,6.0,Yes,No,NaN,No,NaN,Murang'a
9,-0.716643,37.145942,1365.800049,98.400,18-30,Male,Single,0,Unemployed,Less than 10000,No,NaN,17.0,No,No,NaN,No,NaN,Murang'a


In [39]:
healthcare_df.to_excel('../data/processed/final_healthcare.xlsx')